# PGD adversarial training for YOLO26n-OBB on DOTAv1

This notebook builds a 15-class DOTAv1 defense dataset with clean and native-loss PGD images, then fine-tunes `yolo26n-obb.pt`.


In [ ]:
%pip install -q "ultralytics==8.4.48" opencv-python-headless pyyaml matplotlib pandas tqdm


In [ ]:
from pathlib import Path
import gc
import math
import random
import shutil

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from tqdm import tqdm
from ultralytics import YOLO
from ultralytics.cfg import get_cfg
from ultralytics.utils.ops import xyxyxyxy2xywhr

KAGGLE_DATASET_URL = "https://www.kaggle.com/datasets/chandlertimm/dota-data"
DATASET_ROOT = Path("/kaggle/input/dota-data")
MODEL_WEIGHTS = "yolo26n-obb.pt"
IMG_SIZE = 1024
SEED = 42
NUM_EVAL_IMAGES = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_NAMES = {
    0: "plane",
    1: "ship",
    2: "storage tank",
    3: "baseball diamond",
    4: "tennis court",
    5: "basketball court",
    6: "ground track field",
    7: "harbor",
    8: "bridge",
    9: "large vehicle",
    10: "small vehicle",
    11: "helicopter",
    12: "roundabout",
    13: "soccer ball field",
    14: "swimming pool",
}
NAME_TO_ID = {name: idx for idx, name in CLASS_NAMES.items()}
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert len(CLASS_NAMES) == 15
print(f"Device: {DEVICE}")
print(f"Dataset: {KAGGLE_DATASET_URL}")
print(f"Expected Kaggle path: {DATASET_ROOT}")


In [ ]:
def assert_dataset_root() -> None:
    if not DATASET_ROOT.exists():
        raise FileNotFoundError(
            f"Attach Kaggle dataset chandlertimm/dota-data so {DATASET_ROOT} exists. "
            f"Dataset URL: {KAGGLE_DATASET_URL}"
        )


def normalize_class_name(name: str) -> str:
    return name.strip().lower().replace("-", " ")


def parse_label_lines(label_path: Path, width: int, height: int) -> list[str]:
    rows = []
    for raw_line in label_path.read_text().splitlines():
        parts = raw_line.strip().split()
        if not parts:
            continue

        if len(parts) == 9 and parts[0].isdigit():
            cls_id = int(parts[0])
            coords = [float(x) for x in parts[1:9]]
            assert cls_id in CLASS_NAMES, f"Unexpected class id {cls_id} in {label_path}"
            assert all(0.0 <= x <= 1.0 for x in coords), f"YOLO OBB coords must be normalized in {label_path}"
            rows.append(f"{cls_id} " + " ".join(f"{x:.6f}" for x in coords))
            continue

        if len(parts) < 9:
            continue
        if not parts[0].replace(".", "", 1).isdigit():
            continue

        cls_name = normalize_class_name(parts[8])
        if cls_name not in NAME_TO_ID:
            raise ValueError(
                f"Found class '{cls_name}' in {label_path}. This notebook is fixed to DOTAv1's 15 classes."
            )
        coords = [float(x) for x in parts[:8]]
        norm = []
        for idx in range(0, 8, 2):
            norm.append(coords[idx] / width)
            norm.append(coords[idx + 1] / height)
        assert all(math.isfinite(x) for x in norm), f"Non-finite normalized coords in {label_path}"
        rows.append(f"{NAME_TO_ID[cls_name]} " + " ".join(f"{x:.6f}" for x in norm))
    return rows


def collect_image_label_pairs() -> list[tuple[Path, Path]]:
    assert_dataset_root()
    images_by_stem = {}
    for image_path in DATASET_ROOT.rglob("*"):
        if image_path.suffix.lower() in IMAGE_EXTS:
            images_by_stem.setdefault(image_path.stem, image_path)

    pairs = []
    for label_path in DATASET_ROOT.rglob("*.txt"):
        image_path = images_by_stem.get(label_path.stem)
        if image_path is not None:
            pairs.append((image_path, label_path))

    if not pairs:
        raise RuntimeError(f"No image/label pairs found under {DATASET_ROOT}")

    val_pairs = [p for p in pairs if "val" in {part.lower() for part in p[0].parts + p[1].parts}]
    chosen = val_pairs if val_pairs else pairs
    chosen = sorted(chosen, key=lambda x: str(x[0]))
    if val_pairs:
        print(f"Found {len(pairs)} labelled images. Using {len(chosen)} validation-labelled images.")
    else:
        print(f"Found {len(pairs)} labelled images. No explicit val split found, using all labelled images.")
    return chosen


def write_yaml(dataset_dir: Path, yaml_path: Path) -> None:
    yaml_path.write_text(yaml.safe_dump({
        "path": str(dataset_dir.resolve()),
        "train": "images",
        "val": "images",
        "nc": len(CLASS_NAMES),
        "names": CLASS_NAMES,
    }, sort_keys=False))


def prepare_clean_subset(work_dir: Path, num_images: int) -> tuple[Path, Path, list[str]]:
    clean_dir = work_dir / "clean"
    if clean_dir.exists():
        shutil.rmtree(clean_dir)
    (clean_dir / "images").mkdir(parents=True)
    (clean_dir / "labels").mkdir(parents=True)

    pairs = collect_image_label_pairs()
    rng = random.Random(SEED)
    selected = rng.sample(pairs, min(num_images, len(pairs)))

    kept = []
    for image_path, label_path in tqdm(selected, desc="Preparing clean subset"):
        image = cv2.imread(str(image_path))
        assert image is not None, f"Could not read {image_path}"
        height, width = image.shape[:2]
        rows = parse_label_lines(label_path, width, height)
        if not rows:
            continue

        out_name = image_path.stem + ".png"
        resized = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        cv2.imwrite(str(clean_dir / "images" / out_name), resized)
        (clean_dir / "labels" / f"{image_path.stem}.txt").write_text("\n".join(rows) + "\n")
        kept.append(out_name)

    assert kept, "No labelled DOTAv1 images were prepared. Check the dataset structure."
    data_yaml = work_dir / "data_clean.yaml"
    write_yaml(clean_dir, data_yaml)
    print(f"Prepared {len(kept)} clean images in {clean_dir}")
    return clean_dir, data_yaml, kept


def load_model(weights: str = MODEL_WEIGHTS) -> YOLO:
    model = YOLO(weights)
    head = model.model.model[-1]
    assert getattr(head, "nc", None) == 15, f"Expected 15-class DOTAv1 OBB head, got {getattr(head, 'nc', None)}"
    model.model.args = get_cfg(overrides=model.model.args)
    model.model.names = CLASS_NAMES
    model.model.to(DEVICE)
    model.model.eval()
    return model


def read_image_tensor(image_path: Path) -> torch.Tensor:
    bgr = cv2.imread(str(image_path))
    assert bgr is not None, f"Could not read {image_path}"
    bgr = cv2.resize(bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE) / 255.0
    return tensor


def save_tensor_image(tensor: torch.Tensor, image_path: Path) -> None:
    rgb = tensor.detach().clamp(0, 1).squeeze(0).permute(1, 2, 0).cpu().numpy()
    bgr = cv2.cvtColor((rgb * 255).round().astype(np.uint8), cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(image_path), bgr)


def label_batch(label_path: Path, image_tensor: torch.Tensor) -> dict[str, torch.Tensor]:
    rows = []
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if not parts:
            continue
        rows.append([float(x) for x in parts])
    assert rows, f"Empty label file: {label_path}"
    data = torch.tensor(rows, dtype=torch.float32, device=DEVICE)
    cls = data[:, 0]
    polygons = data[:, 1:9]
    bboxes = xyxyxyxy2xywhr(polygons)
    batch_idx = torch.zeros((data.shape[0],), dtype=torch.float32, device=DEVICE)
    return {"img": image_tensor, "cls": cls, "bboxes": bboxes, "batch_idx": batch_idx}


def native_obb_loss(model: YOLO, image_tensor: torch.Tensor, label_path: Path) -> torch.Tensor:
    batch = label_batch(label_path, image_tensor)
    loss_vec, _ = model.model.loss(batch)
    return loss_vec.sum()


def evaluate_model(model: YOLO, data_yaml: Path, label: str) -> dict[str, float | str]:
    metrics = model.val(data=str(data_yaml), task="obb", imgsz=IMG_SIZE, device=DEVICE, plots=False, verbose=False)
    result = {
        "label": label,
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
    }
    print(result)
    return result


def f1(precision: float, recall: float) -> float:
    return 2 * precision * recall / (precision + recall + 1e-9)


def plot_attack_metrics(metrics_csv: Path, plot_dir: Path, title: str) -> None:
    df = pd.read_csv(metrics_csv)
    plot_dir.mkdir(parents=True, exist_ok=True)
    clean = df[df["split"] == "clean"].iloc[0]
    adv = df[df["split"] == "attack"].sort_values("epsilon")

    plt.figure(figsize=(9, 5))
    plt.axhline(clean["map50"], linestyle="--", color="tab:blue", label="Clean mAP50")
    plt.axhline(clean["map50_95"], linestyle="--", color="tab:red", label="Clean mAP50-95")
    plt.plot(adv["epsilon"], adv["map50"], "o-", color="tab:blue", label="Attack mAP50")
    plt.plot(adv["epsilon"], adv["map50_95"], "s-", color="tab:red", label="Attack mAP50-95")
    plt.xlabel("Epsilon")
    plt.ylabel("mAP")
    plt.title(f"{title}: mAP vs epsilon")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_dir / "map.png", dpi=160)
    plt.show()

    plt.figure(figsize=(9, 5))
    plt.axhline(clean["precision"], linestyle="--", color="tab:green", label="Clean precision")
    plt.axhline(clean["recall"], linestyle="--", color="tab:purple", label="Clean recall")
    plt.plot(adv["epsilon"], adv["precision"], "^-", color="tab:green", label="Attack precision")
    plt.plot(adv["epsilon"], adv["recall"], "v-", color="tab:purple", label="Attack recall")
    plt.xlabel("Epsilon")
    plt.ylabel("Score")
    plt.title(f"{title}: precision and recall vs epsilon")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_dir / "precision_recall.png", dpi=160)
    plt.show()

    clean_f1 = f1(clean["precision"], clean["recall"])
    adv_f1 = [f1(p, r) for p, r in zip(adv["precision"], adv["recall"])]
    plt.figure(figsize=(9, 5))
    plt.axhline(clean_f1, linestyle="--", color="black", label="Clean F1")
    plt.plot(adv["epsilon"], adv_f1, "D-", color="black", label="Attack F1")
    plt.xlabel("Epsilon")
    plt.ylabel("F1")
    plt.title(f"{title}: F1 vs epsilon")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_dir / "f1.png", dpi=160)
    plt.show()


def clear_memory() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()


In [ ]:
WORK_DIR = Path("/kaggle/working/yolo26_obb_pgd_defense")
NUM_DEFENSE_IMAGES = 200
NUM_EVAL_IMAGES_DEFENSE = 50
DEFENSE_EPSILON = 0.04
PGD_STEPS = 10
PGD_ALPHA_FRACTION = 0.25
EPOCHS = 25
BATCH = 8
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

def pgd_attack(model: YOLO, image_path: Path, label_path: Path, epsilon: float) -> torch.Tensor:
    original = read_image_tensor(image_path).detach()
    adv = (original + torch.empty_like(original).uniform_(-epsilon, epsilon)).clamp(0, 1).detach()
    alpha = epsilon * PGD_ALPHA_FRACTION
    for _ in range(PGD_STEPS):
        adv.requires_grad_(True)
        loss = native_obb_loss(model, adv, label_path)
        model.model.zero_grad(set_to_none=True)
        loss.backward()
        assert adv.grad is not None, f"No input gradient for {image_path}"
        with torch.no_grad():
            adv = adv + alpha * adv.grad.sign()
            delta = torch.clamp(adv - original, min=-epsilon, max=epsilon)
            adv = torch.clamp(original + delta, min=0, max=1).detach()
    return adv

clean_dir, clean_yaml, image_names = prepare_clean_subset(WORK_DIR, NUM_DEFENSE_IMAGES + NUM_EVAL_IMAGES_DEFENSE)
defense_names = image_names[:min(NUM_DEFENSE_IMAGES, len(image_names))]
attack_model = load_model()
defense_dir = WORK_DIR / "defense_dataset"
(defense_dir / "images" / "train").mkdir(parents=True)
(defense_dir / "labels" / "train").mkdir(parents=True)

for image_name in tqdm(defense_names, desc="Building defense dataset"):
    image_path = clean_dir / "images" / image_name
    label_path = clean_dir / "labels" / f"{Path(image_name).stem}.txt"
    clean_out = defense_dir / "images" / "train" / f"clean_{image_name}"
    adv_out = defense_dir / "images" / "train" / f"pgd_{image_name}"
    shutil.copy(image_path, clean_out)
    shutil.copy(label_path, defense_dir / "labels" / "train" / f"clean_{label_path.name}")
    adv = pgd_attack(attack_model, image_path, label_path, DEFENSE_EPSILON)
    save_tensor_image(adv, adv_out)
    shutil.copy(label_path, defense_dir / "labels" / "train" / f"pgd_{label_path.name}")

defense_yaml = WORK_DIR / "defense_data.yaml"
defense_yaml.write_text(yaml.safe_dump({
    "path": str(defense_dir.resolve()),
    "train": "images/train",
    "val": "images/train",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}, sort_keys=False))

model = load_model()
train_device = [0, 1] if torch.cuda.device_count() >= 2 else 0 if torch.cuda.is_available() else "cpu"
model.train(
    data=str(defense_yaml),
    task="obb",
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=train_device,
    lr0=1e-4,
    freeze=10,
    project=str(WORK_DIR / "training"),
    name="pgd_defended_yolo26n_obb_dotav1",
    exist_ok=True,
)
print(WORK_DIR / "training" / "pgd_defended_yolo26n_obb_dotav1" / "weights" / "best.pt")
